<a href="https://colab.research.google.com/github/yogasgm/prototype_finetuning_pytorch/blob/main/Prototype_Multilabel_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

# Importing libraries

In [ ]:
!pip install transformers

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import random
import shutil
import sys
from sklearn.model_selection import train_test_split

In [ ]:
import pandas as pd
df = pd.read_excel('[INPUT_YOUR_FILE_HERE]')  # <- was: Stage 1 Institutional News.xlsx

# Assuming 'df' from previous cells is your data:
train_df = df.copy()  # Create a copy of 'df' and name it 'train_df'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Setting seed for reproducibility

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
set_seed(43)

# Downloading dataset

In [ ]:
# Load the dataset directly from the Colab file system
file_path = '[INPUT_YOUR_FILE_HERE]'  # Adjust the filename as needed
train_df = pd.read_excel(file_path)

In [ ]:
train_df.info()

In [ ]:
train_df.columns

# Selecting required columns

In [ ]:
train_df = train_df['text', 'Resilience and Adaptive Capacity',
       'Policy Integration and Governance',
       'Awareness and Operational Capacity']

In [ ]:
target_list = ['Resilience and Adaptive Capacity',
       'Policy Integration and Governance',
       'Awareness and Operational Capacity']

# Preparing the tokenizer

In [ ]:
# Set Max Length, maximum 512 (BERT)
MAX_LEN = 512

In [ ]:
from transformers import BertTokenizer, BertModel

In [ ]:
#download the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
class CustomDataset(torch.utils.data.Dataset):

    def __init__(self, df, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.df = df
        self.title = df['text']
        self.targets = self.df[target_list].values
        self.max_len = max_len

    def __len__(self):
        return len(self.title)

    def __getitem__(self, index):
        title = str(self.title[index])
        title = " ".join(title.split())

        inputs = self.tokenizer.encode_plus(
            title,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=True,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'token_type_ids': inputs["token_type_ids"].flatten(),
            'targets': torch.FloatTensor(self.targets[index])
        }

# Splitting & Tokenizing Dataset

In [ ]:
# Adjusting the train/validation/test split
train_df, temp_df = train_test_split(train_df, test_size=0.2, random_state=43)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=43)

# Reset the indices
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
# Label distribution in the training set
train_counts = train_df[target_list].sum(axis=0)
print("Label distribution in the training set:\n", train_counts)

# Label distribution in the validation set
val_counts = val_df[target_list].sum(axis=0)
print("\nLabel distribution in the validation set:\n", val_counts)

# Label distribution in the test set
test_counts = test_df[target_list].sum(axis=0)
print("\nLabel distribution in the test set:\n", test_counts)

In [ ]:
# Label distribution in the training set
train_counts_percentage = (train_df[target_list].sum(axis=0) / len(train_df)) * 100
print("Label distribution in the training set:\n", train_counts_percentage)

# Label distribution in the validation set
val_counts_percentage = (val_df[target_list].sum(axis=0) / len(val_df)) * 100
print("\nLabel distribution in the validation set:\n", val_counts_percentage)

# Label distribution in the test set
test_counts_percentage = (test_df[target_list].sum(axis=0) / len(test_df)) * 100
print("\nLabel distribution in the test set:\n", test_counts_percentage)

In [ ]:
train_df.shape

In [ ]:
val_df.shape

In [ ]:
val_df

In [ ]:
test_df

In [ ]:
# Create the CustomDataset for each set
train_dataset = CustomDataset(train_df, tokenizer, MAX_LEN)
valid_dataset = CustomDataset(val_df, tokenizer, MAX_LEN)
test_dataset = CustomDataset(test_df, tokenizer, MAX_LEN)

In [ ]:
len(train_dataset)

# Setting hyperparameters

In [ ]:
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 2e-5

In [ ]:
# Preparing the DataLoaders
train_data_loader = torch.utils.data.DataLoader(train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

val_data_loader = torch.utils.data.DataLoader(valid_dataset,
    batch_size=VALID_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [ ]:
# Checking for available device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
device

# Additional functions for loading and saving checkpoints

In [ ]:
def load_ckp(checkpoint_fpath, model, optimizer):
    """
    checkpoint_path: path to save checkpoint
    model: model that we want to load checkpoint parameters into
    optimizer: optimizer we defined in previous training
    """
    # load check point
    checkpoint = torch.load(checkpoint_fpath)
    # initialize state_dict from checkpoint to model
    model.load_state_dict(checkpoint['state_dict'])
    # initialize optimizer from checkpoint to optimizer
    optimizer.load_state_dict(checkpoint['optimizer'])
    # initialize valid_loss_min from checkpoint to valid_loss_min
    valid_loss_min = checkpoint['valid_loss_min']
    # return model, optimizer, epoch value, min validation loss
    return model, optimizer, checkpoint['epoch'], valid_loss_min

def save_ckp(state, is_best, checkpoint_path, best_model_path):
    """
    state: checkpoint we want to save
    is_best: is this the best checkpoint; min validation loss
    checkpoint_path: path to save checkpoint
    best_model_path: path to save best model
    """
    f_path = checkpoint_path
    # save checkpoint data to the path given, checkpoint_path
    torch.save(state, f_path)
    # if it is a best model, min validation loss
    if is_best:
        best_fpath = best_model_path
        # copy that checkpoint file to best path given, best_model_path
        shutil.copyfile(f_path, best_fpath)

# Training the Model

Defining and Initializing the BERT Classification Model

In [ ]:
class BERTClass(torch.nn.Module):
    def __init__(self):
        super(BERTClass, self).__init__()
        self.bert_model = BertModel.from_pretrained('bert-base-uncased', return_dict=True)
        self.dropout = torch.nn.Dropout(0.3)
        self.linear = torch.nn.Linear(768, 3)

    def forward(self, input_ids, attn_mask, token_type_ids):
        output = self.bert_model(
            input_ids,
            attention_mask=attn_mask,
            token_type_ids=token_type_ids
        )
        output_dropout = self.dropout(output.pooler_output)
        output = self.linear(output_dropout)
        return output

model = BERTClass()
model.to(device)

Setting Up the Loss Function and Optimizer

In [ ]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

optimizer = torch.optim.AdamW(params =  model.parameters(), lr=LEARNING_RATE)

Initialization of Validation Target and Output Lists

In [ ]:
val_targets=[]
val_outputs=[]

Training and Validation Loop with Early Stopping

In [ ]:
def train_model(n_epochs, training_loader, validation_loader, model,
                optimizer, checkpoint_path, best_model_path, patience):

  # initialize tracker for minimum validation loss
  valid_loss_min = np.inf # Change np.Inf to np.inf
  no_improve = 0


  for epoch in range(1, n_epochs+1):
    train_loss = 0
    valid_loss = 0

    model.train()
    print('############# Epoch {}: Training Start   #############'.format(epoch))
    for batch_idx, data in enumerate(training_loader):
        #print('yyy epoch', batch_idx)
        ids = data['input_ids'].to(device, dtype = torch.long)
        mask = data['attention_mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = model(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        #if batch_idx%5000==0:
         #   print(f'Epoch: {epoch}, Training Loss:  {loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #print('before loss data in training', loss.item(), train_loss)
        train_loss = train_loss + ((1 / (batch_idx + 1)) * (loss.item() - train_loss))
        #print('after loss data in training', loss.item(), train_loss)

    print('############# Epoch {}: Training End     #############'.format(epoch))

    print('############# Epoch {}: Validation Start   #############'.format(epoch))
    ######################
    # validate the model #
    ######################

    model.eval()

    with torch.no_grad():
      for batch_idx, data in enumerate(validation_loader, 0):
            ids = data['input_ids'].to(device, dtype = torch.long)
            mask = data['attention_mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = model(ids, mask, token_type_ids)

            loss = loss_fn(outputs, targets)
            valid_loss = valid_loss + ((1 / (batch_idx + 1)) * (loss.item() - valid_loss))
            val_targets.extend(targets.cpu().detach().numpy().tolist())
            val_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())

      print('############# Epoch {}: Validation End     #############'.format(epoch))
      # calculate average losses
      #print('before cal avg train loss', train_loss)
      train_loss = train_loss/len(training_loader)
      valid_loss = valid_loss/len(validation_loader)
      # print training/validation statistics
      print('Epoch: {} \tAvgerage Training Loss: {:.6f} \tAverage Validation Loss: {:.6f}'.format(
            epoch,
            train_loss,
            valid_loss
            ))

      # create checkpoint variable and add important data
      checkpoint = {
            'epoch': epoch + 1,
            'valid_loss_min': valid_loss,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict()
      }


      ## TODO: save the model if validation loss has decreased
      if valid_loss <= valid_loss_min:
        print('Validation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(
              valid_loss_min,
              valid_loss
              ))
        save_ckp(checkpoint, True, checkpoint_path, best_model_path)
        valid_loss_min = valid_loss
        no_improve = 0
      else:
        no_improve += 1
        if no_improve >= patience:
          print("Early stopping due to no improvement in validation loss")
          break

  return model

In [ ]:
# Save checkpoint

ckpt_path = '/content/ckpt.pth'
best_model_path = '/content/best_model.pth'

# Start Train

In [ ]:
trained_model = train_model(EPOCHS, train_data_loader, val_data_loader, model, optimizer, ckpt_path, best_model_path, patience=2)

In [ ]:
# Load the saved checkpoint
model, optimizer, start_epoch, valid_loss_min = load_ckp(best_model_path, model, optimizer)

print(f'The validation loss of the best saved model is: {valid_loss_min}')

# Test

In [ ]:
# Process new dataset
#new_dataset = CustomDataset(new_df, tokenizer, MAX_LEN)
new_dataset = test_dataset

# Create DataLoader
new_data_loader = torch.utils.data.DataLoader(new_dataset,
    batch_size=VALID_BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

# Load the model
model, optimizer, start_epoch, valid_loss_min = load_ckp(best_model_path, model, optimizer)

# Switch model to the evaluation mode
model.eval()

new_outputs = []
new_targets = []
test_loss = 0.0

# Define loss function
loss_fn = torch.nn.BCEWithLogitsLoss()

# Pass new data through the model
with torch.no_grad():
    for batch_idx, data in enumerate(new_data_loader):
        ids = data['input_ids'].to(device, dtype = torch.long)
        mask = data['attention_mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = model(ids, mask, token_type_ids)

        # Calculate loss
        loss = loss_fn(outputs, targets)
        test_loss += loss.item() * data['input_ids'].size(0)

        new_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
        new_targets.extend(targets.cpu().detach().numpy().tolist())

# Average the test loss over all batches
test_loss = test_loss / len(new_data_loader.dataset)

print(f'Test Loss: {test_loss:.6f}')

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Convert the outputs and targets to numpy arrays
new_outputs_np = np.array(new_outputs)
new_targets_np = np.array(new_targets)

# Threshold the outputs (This depends on your requirements, 0.5 is used as an example)
new_outputs_bin = (new_outputs_np > 0.5)

# Calculate metrics
print(classification_report(new_targets_np, new_outputs_bin))

# Calculate macro and micro metrics
precision_macro = precision_score(new_targets_np, new_outputs_bin, average='macro')
recall_macro = recall_score(new_targets_np, new_outputs_bin, average='macro')
f1_macro = f1_score(new_targets_np, new_outputs_bin, average='macro')

precision_micro = precision_score(new_targets_np, new_outputs_bin, average='micro')
recall_micro = recall_score(new_targets_np, new_outputs_bin, average='micro')
f1_micro = f1_score(new_targets_np, new_outputs_bin, average='micro')

print(f'Macro Precision: {precision_macro} Macro Recall: {recall_macro} Macro F1: {f1_macro}')
print(f'Micro Precision: {precision_micro} Micro Recall: {recall_micro} Micro F1: {f1_micro}')

In [ ]:
from sklearn.metrics import accuracy_score

# Calculate accuracy
accuracy = accuracy_score(new_targets_np, new_outputs_bin)

print(f'Accuracy: {accuracy}')

# **PREDICTION**

In [ ]:
pip install transformers torch


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification


In [ ]:
# Replace with your actual model path and labels count
model_path = '/content/best_model.pth'  # Path to your saved checkpoint
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # Same tokenizer as used in training
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)  # Set num_labels to your class count


In [ ]:
import torch

# Path to your checkpoint file
checkpoint_path = '/content/ckpt.pth'

# Try loading the checkpoint
try:
    state_dict = torch.load(checkpoint_path, map_location=torch.device('cuda'))
    print("Checkpoint loaded successfully.")
    print("Keys:", state_dict.keys())
except Exception as e:
    print(f"Failed to load the checkpoint: {e}")


In [ ]:
import torch

# Load the checkpoint and inspect its contents
checkpoint_path = '/content/ckpt.pth'  # Replace with your checkpoint file path
state_dict = torch.load(checkpoint_path, map_location=torch.device('cuda'))  # Or 'cuda' if using GPU
print(state_dict.keys())


In [ ]:
print(state_dict['state_dict'].keys())


In [ ]:
import torch
from transformers import BertForSequenceClassification

# Path to your checkpoint file
checkpoint_path = '/content/ckpt.pth'

# Initialize the model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)

# Load the checkpoint
state_dict = torch.load(checkpoint_path, map_location=torch.device('cuda'))  # Use 'cuda' if using GPU

# Access the nested 'state_dict'
# This assumes your checkpoint saved the model's state_dict under the key 'state_dict'
state_dict = state_dict['state_dict']

# Load the model weights directly
model.load_state_dict(state_dict, strict=False) # strict=False to ignore unexpected keys

# Set the model to evaluation mode
model.eval()

print("Model successfully loaded and ready for evaluation.")

In [ ]:
import numpy as np
def predict_in_batches(texts, batch_size=8):
    # Initialize lists to store predictions
    all_predictions = []

    # Process texts in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Tokenize the batch
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )

        # Move to device (GPU/CPU)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model.to(device)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Make predictions
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            batch_predictions = torch.sigmoid(logits)

        # Move predictions to CPU and convert to numpy
        batch_predictions = batch_predictions.cpu().numpy()
        all_predictions.extend(batch_predictions)

        # Clear GPU cache if using CUDA
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Optional: Add a progress indicator
        if i % (batch_size * 10) == 0:
            print(f"Processed {i}/{len(texts)} texts")

    return np.array(all_predictions)



In [ ]:
texts = [
    "Reform UK council leader in Northants criticised on net zero Reform council leader criticised on net zero stance 30 May 2025 Share Save James Grant BBC News, Northamptonshire and Nadia Lincoln Local Democracy Reporter Service Share Save PA Media Martin Griffiths, the new leader of North Northamptonshire Council, has said net zero was not a matter for local councils A newly elected Reform UK council leader has been criticised over his remarks about net zero targets. Earlier this week, North Northamptonshire Council's Martin Griffiths questioned the role of local authorities in tackling climate change. Griffiths, appointed council leader on 22 May, said his party was not made up of climate change deniers but believed that net zero was a global matter that was making everyone poorer. But the leader of the Green Party group on the council, Emily Fedorowycz, said his statements on net zero were irresponsible and dangerous. Emily Fedorowycz said there were economic benefits and job opportunities in renewable energy Fedorowycz said there was an enormous economic opportunity from climate projects and there would be future costs to residents if targets were ignored. Blaming climate action for rising poverty is a deliberate distraction from the real causes: a broken energy system, years of underinvestment in insulation and public transport, and global reliance on fossil fuels. Laura Coffey/BBC Martin Griffiths is the new leader of Reform-run North Northamptonshire Council Labour MP for Kettering Rosie Wrighting also criticised the comments, writing on social media: Constituents regularly raise concerns with me about the climate emergency and bills. Tackling net zero is vital to both issues, so I am disappointed to read these comments by the new Reform leader of [North Northamptonshire Council] Wrighting added: Councillor Griffiths describes [net zero] as a global matter, brushing it off as someone elses problem. But local councils have a role to play and I will be watching the councils climate approach closely. House of Commons Rosie Wrighting MP took to social media platform X to criticise the new leader of North Northamptonshire Council North Northamptonshire Council declared a climate emergency in 2021 and committed to becoming a carbon-neutral authority by 2030. The councils targets align with the UK's national legal requirement to reach net zero emissions by 2050 or earlier. The Reform administration is expected to lead the council until 2029, shaping local environmental and policy decisions during that time. PA Media Martin Griffiths comments mirror the Reform Partys national stance, led by Nigel Farage Speaking to the Local Democracy Reporting Service earlier this week, Griffiths said on other areas relating to the climate and environment, he agreed with wanting to plant more trees and clean up the countys rivers. The stance taken by the new leader of the council mirrors the opinions of the Reform party at the national level. Griffiths has been contacted for further comment. Dr Iain Staffell of Imperial College London said that in the long term, renewables will be bringing down overall energy bills, and specifically electricity bills, for the UK. But some analysts have said in the short term, green energy prices could rise due to the rush to secure enough renewables to meet the net zero goal. Follow Northamptonshire news on BBC Sounds, Facebook, Instagram and X",
    "Heat pumps: How do they work, what do they cost and are they noisy? What are heat pumps and how much do they cost? 29 May 2025 Share Save Share Save Andrew Aitchison/Getty Images A planning restriction that prevented heat pumps being installed within a metre of a neighbour's property has been removed. The government hopes the move will encourage more people to install the low-carbon technology. However, installation rates will need to increase substantially if the government wants to meet its target of 600,000 heat pumps being fitted each year by 2028. Planning change to make installing heat pump easier for millions What are heat pumps and how do they work? Heat pumps run on electricity instead of gas. They warm buildings by absorbing and amplifying heat from the air, ground, or water. They are widely seen as the best way of cutting emissions of carbon dioxide - a planet-warming gas - from home heating, which accounts for 14% of the UK's carbon emissions. Heat pumps are more efficient than gas boilers and can use electricity generated from increasingly clean sources, as wind and solar power replace polluting fossil fuels. Air-source pumps - the most common type - suck in outdoor air and pass it over tubes containing refrigerant fluids which concentrate and boost the warmth to produce heat. The system consists of a box measuring about 1m x 1m x 0.4m which stands outside the property, as well as a heat pump unit and hot water cylinder inside the property. The indoor unit is about the size of a gas boiler, while the cylinder depends on the size of the home. Ground-source heat pumps are more efficient than air-source models. However, they are typically more expensive and less commonly used, as they require either a deep bore hole or a horizontal system dug into the ground over a large area. How much do heat pumps cost? Could a heat pump save me money? While the upfront costs are currently substantial, heat pumps could become cheaper to run than gas boilers, according to the Climate Change Committee (CCC), which advises the UK government on cutting emissions. The cost depends on individual energy prices and how efficiently the heat pump works. Electric heat pumps use much less energy than gas boilers, but electricity typically costs more than gas. Energy deals designed for heat pump owners can also help households make savings. The CCC has called on the government to prioritise making electricity cheaper for everyone, which would make heat pumps more attractive. Andrew Aitchison/Getty Images The rule requiring planning permission if you wanted a heat pump within 1m of your neighbour's property has now been dropped to increase uptake Are heat pumps noisy? Previously, homeowners needed planning permission if they wanted to put a heat pump within one metre of their neighbour's property - because of concerns over noise. The rule was dropped in May to accelerate the uptake of heat pumps. Concerns over noise are also less of an issue with newer devices, though units will still be required to be below a certain volume level. The level has been set at 42db which is a similar output to that of a fridge. The planning changes also include a relaxation of the rules for the size and number of heat pumps households can install. How many heat pumps have been installed in the UK? Rates of heat pump installation in the UK are lower than in other major European countries, such as France, Germany and Italy. But sales are increasing. Nearly 100,000 heat pumps were sold in 2024, up from about 60,000 in 2023, according to the Heat Pump Association. However, the CCC says this number needs to rise to nearly 450,000 a year by 2030 and 1.5 million by 2035 to help meet climate targets. It says around half of UK homes need to have heat pumps by 2040. Significantly more trained heat pump installers are needed to achieve this. Do I have to replace my gas boiler? There is no requirement to replace your existing boiler before the end of its life. Households can still buy a new gas boiler if they wish. However, the CCC recommends that all new home heating should be low-carbon after 2035. Most of this will mean using heat pumps, but it acknowledges that other approaches may be more appropriate in some cases - such as direct electric heating in homes with lower heat demand. But the CCC wants the government to rule out the possible use of hydrogen in home heating to provide certainty to customers and industry."
]


In [ ]:
predictions = predict_in_batches(texts, 8)


In [ ]:
for i, text in enumerate(texts):
    print(f"Text: {text}")
    print(f"Predictions: {predictions[i].tolist()}")  # Convert tensor to list for easier readability


In [ ]:
import pandas as pd

# Load the dataset
dataset_path = '[INPUT_YOUR_FILE_HERE]'  # Update with the path to your dataset
df = pd.read_excel(dataset_path)

# Display the first few rows of your dataset
print(df.head())

# Use the correct column name based on the printed head
texts = df[' Kent County Council rescinds climate change emergency declaration Council rescinds climate change emergency declaration 18 September 2025 Share Save Bob Dale South East Share Save Fiona Irving/BBC Protesters against the motion gathered outside County Hall in Maidstone before the meeting Kent County Council has rescinded its declaration of a climate emergency. The motion was proposed by Reform UK members after the party took control in May\'s local elections. Protesters from both sides of the debate gathered outside County Hall in Maidstone on Thursday, leading to a short confrontation between members of the two groups. The motion, which was passed by 50 votes to 21, with three abstentions, means the council will no longer have to consider net zero ambitions when making decisons. The declaration was passed in 2019, with an ambition to reach net zero greenhouse gas emissions by 2050. Sarah Waite-Gleave, who was protesting the move, said: "We are getting more floods, the Kent Resilience forum are asking all towns to have plans in place to protect people from extreme weather conditions, which are made more frequent by climate change. "To deny we need that sort of resilience for our communities is completely daft." Fiona Irving/BBC Some locals showed support of the Reform UK motion The motion said the declaration "has had no discernible effect on the world\'s climate" and the resulting policies have been "to the detriment of small, local suppliers" and "scared numerous young people". The Conservative Party, who lost control of the council in May, said it would support the Reform UK motion, as did the single UKIP member. The Green Party, Liberal Democrats and Labour Party opposed it. Michael Keohan/BBC Members of Kent County Council voted to rescind the declaration Antony Hook, the leader of the Liberal Democrat opposition, told BBC Radio Kent: "I think it\'s political vandalism from Reform. "Ask any farmer in Kent and they\'ll tell you their crop yields are going down because of the very hot weather and droughts. We\'ve had wildfires and flash flooding. "Their climate denial motion today really puts Kent at more risk." But council leader Linden Kemkaran said: "For so many years the opposition have been able to shut down anybody who had an opposing point of view. They got so used to shouting across the chamber, \'climate change deniers, lunatics\'. "We\'re not saying there\'s no such thing as climate change, what we are saying is we need to adapt to the changing climate. "Nature will do what nature does." PA Media Kent County Council leader Linden Kemkaran said humans needed to adapt to climate change The overwhelming consensus among scientists is that human activities are causing climate change, posing serious threats to people and nature. The year 2024 was the hottest on record and the first to surpass 1.5C (2.7F) of warming, according to the European Copernicus climate service, one of the main global data providers. The UN\'s climate body - the Intergovernmental Panel on Climate Change (IPCC) - concluded in 2023 that "human activities, principally through emissions of greenhouse gases, have unequivocally caused global warming". Follow BBC Kent on Facebook, on X, and on Instagram. Send your story ideas to southeasttoday@bbc.co.uk or WhatsApp us on 08081 002250.'] # Replace with the actual column name from your dataset

In [ ]:
# Use the function with your data
batch_size = 8  # Adjust this based on your available RAM
predictions = predict_in_batches(texts, batch_size=batch_size)
threshold = 0.55
binary_predictions = (predictions > threshold).astype(int)

In [ ]:
# Convert predictions to a list of lists
df['predictions'] = [pred.tolist() for pred in predictions]


In [ ]:
# Add predictions to the dataframe and save
df['predicted_labels'] = binary_predictions.tolist()
df.to_csv('/content/predicted_dataset.csv', index=False)
print("Predictions saved to '/content/predicted_dataset.csv'")

# Test with New Input Text

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import pandas as pd


In [ ]:
model_name = "bert-base-uncased"  # Replace with your model
model = BertForSequenceClassification.from_pretrained(model_name)
tokenizer = BertTokenizer.from_pretrained(model_name)

# Load model onto GPU or CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


In [ ]:
class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            pad_to_max_length=True,
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }


In [ ]:
# Load test dataset
file_path = "[INPUT_YOUR_FILE_HERE]"  # Path to your test dataset
test_data = pd.read_excel(file_path)
texts = test_data["text"].tolist()  # Replace 'text' with the column name in your dataset

# Create DataLoader
MAX_LEN = 512
BATCH_SIZE = 16
test_dataset = TestDataset(texts, tokenizer, MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
test_texts = df['text'].astype(str).tolist()


In [ ]:
# Example: Preprocessing test data
import pandas as pd

# Load your dataset
df = pd.read_excel('[INPUT_YOUR_FILE_HERE]')  # <- was: Stage 1 Institutional News.xlsx

# Select the text column and ensure it's in string format
texts = df['text'].astype(str).tolist()  # Replace 'text_column' with the actual column name containing text

# Create DataLoader or preprocess texts directly
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128

# Tokenize the texts
def tokenize_function(text):
    return tokenizer(text, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')

tokenized_texts = [tokenize_function(text) for text in texts]

# Create a DataLoader if needed
test_loader = DataLoader(tokenized_texts, batch_size=16)

In [ ]:
def predict(model, data_loader):
    model.eval()
    predictions = []

    with torch.no_grad():
        for batch in data_loader:
            # Move 'input_ids' and 'attention_mask' to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Provide token_type_ids if they are present in the batch
            # otherwise, create a tensor of zeros with the same shape as input_ids
            if 'token_type_ids' in batch:
                token_type_ids = batch['token_type_ids'].to(device)
            else:
                token_type_ids = torch.zeros_like(input_ids, device=device)

            # Pass 'input_ids', 'attention_mask', and 'token_type_ids' to the model
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)

            logits = outputs.logits
            probs = torch.nn.functional.softmax(logits, dim=-1)
            predicted_classes = torch.argmax(probs, dim=-1)
            predictions.extend(predicted_classes.cpu().numpy())

    return predictions

In [ ]:
def predict(model, data_loader):
    for batch in data_loader:
        if isinstance(batch, dict):
            input_ids = batch['input_ids']
            attention_mask = batch['attention_mask']
        else:
            input_ids, attention_mask = batch[:2]  # if tuple or list
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)


In [ ]:
# ipython-input-85-a0dc5aae3232
predicted_labels = predict(model, test_loader)

In [ ]:
# Add predictions to the original dataframe
test_data["predicted_label"] = predicted_labels

# Save to CSV
test_data.to_csv("/content/test_predictions.csv", index=False)
print("Predictions saved to 'test_predictions.csv'")


In [ ]:
def classify_text(model, text, tokenizer, max_len, threshold=0.5):
    # Prepare the text
    inputs = tokenizer.encode_plus(
        text,
        None,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        return_token_type_ids=True,
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    token_type_ids = inputs["token_type_ids"].to(device)

    # Get the model outputs
    with torch.no_grad():
        outputs = model(input_ids, attention_mask, token_type_ids)

    # Convert to probabilities
    probabilities = torch.sigmoid(outputs).cpu().detach().numpy().tolist()

    # Define the class labels in the same order that the model was trained on
    class_labels = ['Excessive Resource Consumption', 'Waste Mismanagement', 'Plastic Pollution', 'Fossil Fuel Dependence', 'Food Waste']


    # Convert the probabilities to labels
    predicted_labels = [class_labels[i] for i, prob in enumerate(probabilities[0]) if prob > threshold]

    return probabilities, predicted_labels


# **PREDICTION**

In [ ]:
! pip install torch transformers pandas matplotlib openpyxl


In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
import matplotlib.pyplot as plt

# Set the device to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
import torch
from transformers import BertForSequenceClassification

# Define device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model architecture
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)

# Load model weights
model_path = "/content/best_model.pth"  # Ensure the path and file are correct
try:
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    print("Model loaded successfully!")
except Exception as e:
    print(f"Failed to load model: {e}")


In [ ]:
def predict(texts, model, tokenizer, batch_size=16):
    predictions = []

    # Process the data in batches
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        # Tokenize the input texts
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,  # Adjust based on your training setup
            return_tensors="pt"
        ).to(device)

        # Predict probabilities
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.sigmoid(logits).cpu().numpy()  # Use sigmoid for probabilities

        predictions.extend(probs)

    return np.array(predictions)


In [ ]:
# Load the unlabelled dataset
dataset_path = "[INPUT_YOUR_FILE_HERE]"  # Replace with your dataset path
df = pd.read_excel(dataset_path)
texts = df['text'].tolist()  # Replace 'text' with the name of your text column


In [ ]:
# Predict on the dataset
predictions = predict(texts, model, tokenizer)


In [ ]:
# Define your labels
labels = ['Excessive Resource Consumption','Waste Mismanagement','Plastic Pollution','Fossil Fuel Dependence','Food Waste',]  # Replace with your labels

# Add predictions as new columns
for i, label in enumerate(labels):
    df[label] = predictions[:, i]

# Save the updated dataset
output_path = "predicted_dataset.xlsx"
df.to_excel(output_path, index=False)
print(f"Predictions saved to: {output_path}")


In [ ]:
def plot_spider_chart(predictions, labels, title="Prediction Visualization"):
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]  # Complete the circle

    # Prepare the data for plotting
    values = predictions.tolist()
    values += values[:1]  # Complete the circle

    # Plot the spider chart
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.fill(angles, values, color='blue', alpha=0.25)
    ax.plot(angles, values, color='blue', linewidth=2)
    ax.set_yticks([])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_title(title, size=16, pad=20)
    plt.show()

# Visualize the first prediction
plot_spider_chart(predictions[0], labels, title="Prediction for First Text")
